# OpticalLayout デモンストレーション

`gtrace.layout.OpticalLayout` は、光学系全体（光学素子・光源・追跡ルール）を表すモデルクラスです。
GUIフロントエンドはこのクラスと1対1対応し、GUIでの編集はこのクラスへの変更として反映されます。

想定するワークフローは2フェーズです:

1. **構築フェーズ**: 通常のPythonコードで光学素子を配置・アラインメントする（逐次追跡や固有モード計算はここで行う）
2. **登録後**: 完成した素子と光源を `OpticalLayout` に登録し、非逐次追跡 (`non_seq_trace`) で可視化・微調整・迷光確認を行う

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # リポジトリのルートを優先

import numpy as np

import gtrace.beam as beam
import gtrace.optcomp as opt
import gtrace.optics.gaussian as gauss
import gtrace.draw.renderer as renderer
from gtrace.layout import OpticalLayout, TraceRules
from gtrace.unit import *

pi = np.pi

## 1. 構築フェーズ

従来通りのPythonコードで、光源ビームと3枚のミラーからなる光学系を構築します。
実際の作業では、ここで逐次追跡（`hitFromHR` など）を使ってアラインメントを追い込みます。

In [2]:
# 光源: ウェスト半径1mmのビームを原点からx方向へ
b0 = beam.GaussianBeam(q0=gauss.Rw2q(np.inf, 1*mm), wl=1064*nm,
                       pos=[0, 0], dirAngle=0, name='b0')

# 3枚のミラー (M1・M2で光路を90度ずつ折り曲げ、M3で逆反射させる)
M1 = opt.Mirror(HRcenter=[0.5, 0.0], normAngleHR=deg2rad(180-45),
                diameter=10*cm, thickness=5*cm, wedgeAngle=deg2rad(0.25),
                inv_ROC_HR=0.0, Refl_HR=0.99, Trans_HR=0.01,
                Refl_AR=500*ppm, Trans_AR=1-500*ppm, n=1.45, name='M1')
M2 = opt.Mirror(HRcenter=[0.5, 0.4], normAngleHR=deg2rad(-90+45),
                diameter=10*cm, thickness=5*cm, wedgeAngle=deg2rad(0.25),
                inv_ROC_HR=1.0/2.0, Refl_HR=0.99, Trans_HR=0.01,
                Refl_AR=500*ppm, Trans_AR=1-500*ppm, n=1.45, name='M2')
M3 = opt.Mirror(HRcenter=[0.9, 0.4], normAngleHR=deg2rad(180),
                diameter=10*cm, thickness=5*cm, wedgeAngle=deg2rad(0.25),
                inv_ROC_HR=1/1.0, Refl_HR=0.9, Trans_HR=0.1,
                Refl_AR=500*ppm, Trans_AR=1-500*ppm, n=1.45, name='M3')

In [3]:
b0.width(3)

(np.float64(0.0014256043491974921), np.float64(0.0014256043491974921))

## 2. OpticalLayout への登録と追跡

完成した素子・光源を `OpticalLayout` に登録します。追跡の振る舞いは `TraceRules` で指定します。

- 素子・光源は**参照で**保持されます（コピーされない）
- 名前はレイアウト内で一意でなければなりません（重複登録は `ValueError`）
- `trace()` は光源をコピーしてから追跡するので、登録した光源オブジェクト自体は変化しません

In [4]:
layout = OpticalLayout(optics=[M1, M2, M3], sources=[b0],
                       rules=TraceRules(order=5, power_threshold=1e-4))

beams = layout.trace()
print(f'追跡されたビームの数: {len(beams)}')
print(f'光源ごとの内訳: {[(k, len(v)) for k, v in layout.beams_by_source.items()]}')
print()
print(f"{'name':>6} {'P [W]':>10} {'stray':>6} {'length [m]':>11}")
for b in beams:
    print(f'{b.name:>6} {b.P:>10.3e} {b.stray_order:>6d} {b.length:>11.4f}')

追跡されたビームの数: 16
光源ごとの内訳: [('b0', 16)]

  name      P [W]  stray  length [m]
    b0  1.000e+00      0      0.5000
 M1:s1  1.000e-02      1      0.0571
 M1:r1  9.900e-01      0      0.4000
 M2:s1  9.900e-03      1      0.0567
 M2:r1  9.801e-01      0      0.4000
 M3:s1  9.801e-02      1      0.0487
 M3:r1  8.821e-01      0      0.4000
 M2:s1  8.821e-03      1      0.0564
 M2:r1  8.733e-01      0      0.4000
 M1:s1  8.733e-03      1      0.0574
 M1:r1  8.645e-01      0      1.0000
 M1:t1  8.728e-03      0      1.0000
 M2:t1  8.816e-03      0      1.0000
 M3:t1  9.796e-02      0      1.0000
 M2:t1  9.895e-03      0      1.0000
 M1:t1  9.995e-03      0      1.0000


## 3. 描画

`layout.draw()` はシーングラフ（`Canvas`）を構築して返します。
ここではDXFに書き出しますが、ブラウザで見るHTMLビューアへの書き出しは 8. を参照してください。

In [5]:
cnv = layout.draw()
renderer.renderDXF(cnv, 'OpticalLayout_demo.dxf')
print('レイヤ:', list(cnv.layers.keys()))
print('OpticalLayout_demo.dxf に書き出しました')

レイヤ: ['main_beam', 'main_beam_width', 'stray_beam', 'stray_beam_width', 'Mirrors', 'text']
OpticalLayout_demo.dxf に書き出しました


## 4. 素子ごとの追跡ルール (max_stray_order)

`order` は、ビームが素子に当たったときに**基板内部の迷光（ゴースト）を何次まで追うか**を決めます。実装上は `beam.stray_order <= order` が保持条件で、この本数は素子ごとに指数的に増えます。

どこまで追う価値があるかは、その素子が光学系で果たす役割に属する性質です（ビームスプリッタのゴーストは効くが、単なる折り返しミラーのゴーストは要らない、など）。そこで**素子自身が `max_stray_order` を持ちます**。`None` ならレイアウト全体の `order` に従い、値が入っていればその素子だけ上書きします。

これは `term_on_HR` / `term_on_HR_order` と同じ場所・同じ考え方です。素子に載っているので、コピーしても、保存・読み込みしても、名前を変えても設定が付いてきます。プロパティパネルの `Max stray order` 欄からも編集でき、空欄・`auto` と入力すると未設定に戻ります。

In [6]:
lay = OpticalLayout(optics=[M1, M2, M3], sources=[b0],
                    rules=TraceRules(order=5, power_threshold=1e-8))
print(f'全素子とも未設定 (order=5):   {len(lay.trace()):>3} beams')

M1.max_stray_order = 0        # M1だけ内部反射を追わない
lay.beams = None
print(f'M1.max_stray_order = 0:      {len(lay.trace()):>3} beams')

# 素子に載っているので、名前を変えても付いてくる
lay.apply_edit({'op': 'rename', 'target': 'M1', 'name': 'PRM'})
print(f'リネーム後も有効:             {len(lay.trace()):>3} beams  '
      f'(PRM.max_stray_order = {lay.get_optics("PRM").max_stray_order})')

lay.apply_edit({'op': 'rename', 'target': 'PRM', 'name': 'M1'})
M1.max_stray_order = None     # 元に戻す

全素子とも未設定 (order=5):    33 beams
M1.max_stray_order = 0:       25 beams
リネーム後も有効:              25 beams  (PRM.max_stray_order = 0)


## 5. 参照セマンティクス — GUIドラッグの基盤

レイアウトは素子を参照で保持しているので、登録済みミラーの属性を書き換えて `trace()` し直すだけで結果が追従します。
将来のGUIでは「ミラーをドラッグ → `HRcenter` を書き換え → 再追跡 → 再描画」という流れになります。

In [7]:
print(f'移動前: 最初のビームの長さ = {layout.trace()[0].length:.3f} m')

M1.HRcenter = [0.6, 0.0]   # ミラーを10cm移動（GUIのドラッグに相当）
print(f'移動後: 最初のビームの長さ = {layout.trace()[0].length:.3f} m')

M1.HRcenter = [0.5, 0.0]   # 元に戻す
print(f'復帰後: 最初のビームの長さ = {layout.trace()[0].length:.3f} m')

移動前: 最初のビームの長さ = 0.500 m
移動後: 最初のビームの長さ = 0.600 m
復帰後: 最初のビームの長さ = 0.500 m


## 6. レイアウトの保存と読み込み

レイアウト（素子・光源・ルール）はJSONファイルとして保存・復元できます。追跡結果は保存されません（`trace()` で再生成できるため）。

In [8]:
layout.save('OpticalLayout_demo_layout.json')

loaded = OpticalLayout.load('OpticalLayout_demo_layout.json')
print(f'読み込んだレイアウト: 素子 {len(loaded.optics)} 個, 光源 {len(loaded.sources)} 個')
print(f'再追跡: {len(loaded.trace())} beams (元のレイアウトと同数)')

with open('OpticalLayout_demo_layout.json') as f:
    print()
    print('JSONファイルの冒頭:')
    print(f.read()[:400], '...')

読み込んだレイアウト: 素子 3 個, 光源 1 個
再追跡: 16 beams (元のレイアウトと同数)

JSONファイルの冒頭:
{
 "name": "Layout",
 "optics": [
  {
   "type": "Mirror",
   "name": "M1",
   "HRcenter": [
    0.5,
    0.0
   ],
   "normAngleHR": 2.356194490192345,
   "diameter": 0.1,
   "thickness": 0.05,
   "wedgeAngle": 0.004363323129985824,
   "inv_ROC_HR": 0.0,
   "inv_ROC_AR": 0.0,
   "Refl_HR": 0.99,
   "Trans_HR": 0.01,
   "Refl_AR": 0.0005,
   "Trans_AR": 0.9995,
   "n": 1.45,
   "HRtransmissive": f ...


## 7. scene_dict — GUIビューアが読むデータ

`layout.scene_dict()` は、描画図形（canvas）とビームの物理パラメータ（beams）をまとめたJSON互換の辞書を返します。
Stage 1以降のHTML/JSビューアはこのデータを読み、ビーム上の任意点クリックで `q' = q + d` からビーム半径やROCを計算して表示します。

In [9]:
sd = layout.scene_dict()
print('トップレベルのキー:', list(sd.keys()))
print('レイヤ:', [ly['name'] for ly in sd['canvas']['layers']])
print()
print('最初のビームのメタデータ:')
for k, v in sd['beams'][0].items():
    print(f'  {k}: {v}')

トップレベルのキー: ['canvas', 'beams', 'optics', 'display']
レイヤ: ['main_beam', 'main_beam_width', 'stray_beam', 'stray_beam_width', 'Mirrors', 'text']

最初のビームのメタデータ:
  name: b0
  layer: main_beam
  pos: [0.0, 0.0]
  end: [0.5, 0.0]
  dirVect: [1.0, 0.0]
  dirAngle: 0.0
  length: 0.5
  wl: 1.064e-06
  n: 1.0
  P: 1.0
  qx: [0.0, 2.952624674426497]
  qy: [0.0, 2.952624674426497]
  wx: 0.001
  wy: 0.001
  Gouyx: 0.0
  Gouyy: 0.0
  optDist: 0.0
  stray_order: 0


## 8. HTMLビューアへの書き出し

`layout.render_html(filename)` は、描画データとビームの物理パラメータを埋め込んだ**自己完結HTML**を1枚書き出します。
外部ファイル・CDN・サーバに一切依存しないので、ダブルクリックで開けますし、共同研究者にそのまま送れます。
DXFを書き出してCADで開く従来のワークフローを、そのまま置き換えるものです。

ブラウザ上でできること:

- ホイールでカーソル中心ズーム、ドラッグでパン
- ビーム上にカーソルを置くと、**その点での**ビームパラメータ（w, ROC, q, ウェスト径と位置, Rayleigh長, Gouy位相, パワー, 光路長）がライブ表示される
- クリックで読み出しを固定。同じ場所を続けてクリックすると、重なったビーム（同じ光路を逆向きに戻るビームなど）を順に切り替えられる
- レイヤごとの表示/非表示

`layout.show()` は同じHTMLを書き出して既定のブラウザで開きます（ファイル名を省略すると一時ファイルを使います）。

In [10]:
html_file = layout.render_html('OpticalLayout_demo.html', title='OpticalLayout demo')
print(f'{html_file} に書き出しました ({os.path.getsize(html_file)/1024:.0f} kB)')

# 自己完結の確認: 外部を参照する src / href が1つも無いこと
import re
with open(html_file, encoding='utf-8') as f:
    doc = f.read()
print('外部参照:', re.findall(r'(?:src|href)\s*=\s*["\']([^"\']+)', doc) or 'なし')

# ブラウザで開くには:
# layout.show()

OpticalLayout_demo.html に書き出しました (132 kB)
外部参照: なし


## 9. ノートブック内での表示（Stage 2）

`layout.widget()` は、同じビューアをノートブックのセル出力に埋め込んだウィジェットを返します（anywidget 使用）。
HTMLファイルを書き出してブラウザに切り替える必要がなくなり、操作性は 8. とまったく同じです。

`layout.show()` は環境に応じて自動で選びます:

| 実行環境 | 選ばれるバックエンド |
|---|---|
| Jupyter カーネル内（anywidget あり） | ウィジェット |
| スクリプト・その他 | 自己完結HTML + ブラウザ起動 |

明示するときは `layout.show(backend='widget')` / `layout.show(backend='html')` と書きます。

ウィジェットはシーンを traitlet として保持しているので、**表示したまま再追跡して描き直せます**。
`w.update()` はレイアウトを追跡し直して新しいシーンを送り込みますが、**ズーム・パン・レイヤの表示状態はそのまま保たれます**。
これが将来の双方向編集（GUIでミラーをドラッグ → Python側で再追跡 → 表示更新）の下地になります。

In [11]:
w = layout.widget(height=560)
w

<gtrace.draw.viewer.widget._build_class.<locals>.LayoutViewer object at 0x000001D68D292270>

In [12]:
# 上のセルを表示したまま、この下のセルを実行すると表示が更新される。
# GUIでミラーをドラッグする操作は、結局これと同じことをする。

M1.HRcenter = [0.52, 0.0]          # ミラーを2cm移動
w.update()
print(f'M1を2cm移動:              {len(w.scene["beams"]):>3} beams')

layout.rules.power_threshold = 1e-6  # 追跡ルールの変更も反映される
w.update()
print(f'power_threshold を下げる: {len(w.scene["beams"]):>3} beams')


M1を2cm移動:               12 beams


power_threshold を下げる:  19 beams


In [13]:

M1.HRcenter = [0.5, 0.0]
layout.rules.power_threshold = 1e-4
w.update()
print(f'元に戻す:                 {len(w.scene["beams"]):>3} beams')

元に戻す:                  16 beams


## 10. 双方向編集（Stage 2b）

9. のウィジェットでは、**ミラーを直接ドラッグして動かせます**。

| 操作 | 動作 |
|---|---|
| 素子をクリック | 右パネルに**プロパティを表示**（ビーム読み出しと切り替わる） |
| 素子をドラッグ | 移動 |
| Shift + ドラッグ | HR面中心まわりの回転 |
| プロパティ欄を編集 | その値をレイアウトに反映 |
| それ以外の場所をドラッグ | 従来どおりパン |

### プロパティパネル

素子をクリックすると、右パネルが「Beam readout」から「Optics properties」に切り替わり、
位置・角度・寸法・ROC・屈折率・反射率が編集可能な欄として表示されます。書き換えて Enter（またはフォーカスを外す）と、その場でレイアウトに反映されます。

ユーザーが考える単位とモデルが持つ traits は一致しないので、パネル側で変換しています:

| 表示 | 実際の trait |
|---|---|
| Angle [°] / Wedge [°] | `normAngleHR` / `wedgeAngle`（ラジアン） |
| ROC HR / ROC AR [m] | `inv_ROC_HR` / `inv_ROC_AR`（逆数。`inf` と入力すると平面 = 0） |
| Center x, y [m] | `center`（基板の中心） |

**回転すると Center が動きます。** 回転は HR 面中心まわりなので基板が振れ、その中心である `center` が移動するためです（`Diameter` や `Wedge` を変えても sag 経由で動きます）。モデルが自己整合を保っている結果で、パネルには往復後の実際の値が表示されます。

数値として解釈できない入力は送られず、欄はモデルの値に戻ります。現在の値と同じ値を入れた場合も何も送りません。

### メッセージ

ドラッグ中は破線のゴーストが追従するだけで、**離した時に1通だけ**メッセージが送られます。プロパティ欄の編集も1欄につき1通です:

```json
{"op": "move",   "target": "M1", "center": [0.52, -0.018]}
{"op": "rotate", "target": "M1", "normAngleHR": 2.269}
{"op": "set",    "target": "M1", "attrs": {"inv_ROC_HR": 0.2857}}
```

これを `OpticalLayout.apply_edit()` が受け、登録済みオブジェクトの traits を書き換え → 再追跡 → 新しいシーンを送り返します。
素子は**参照で**保持されているので、動くのはあなたのコード中の `M1` そのものです。

Pythonが唯一のモデルであり、JS側はビームがどうなるかを一切推測しません。送るのは編集メッセージだけ、返ってくるのは完成したシーンだけです。この分担のまま、Stage 3 では輸送層を websocket に差し替えます。

**編集できる属性はホワイトリスト方式**です（`gtrace.layout.EDITABLE_OPTIC_ATTRS`）。メッセージはブラウザから来るので、`setattr` が通る任意の属性を許すわけにはいきません。拒否された編集はレイアウトに触れず、`error` traitlet 経由でビューア上に表示されます。

なお静的HTML（8.）でも**素子をクリックしてプロパティを読むことはできます**。背後に Python が居ないので編集欄にはならず、読み取り専用の表示になります。

In [14]:
from gtrace.layout import EDITABLE_OPTIC_ATTRS

c0 = np.asarray(M1.center).copy()
print(f'編集前:   M1.center = {c0.round(4)}')

# ビューアでドラッグしたときに送られるのと同じメッセージを手で送る
w.apply_edit({'op': 'move', 'target': 'M1', 'center': [c0[0], c0[1] + 0.01]})
print(f'move後:   M1.center = {np.asarray(M1.center).round(4)}'
      f'   ({len(w.scene["beams"])} beams)')

# 許可されていない属性は拒否され、レイアウトには一切触れない
w.apply_edit({'op': 'set', 'target': 'M1', 'attrs': {'name': 'hacked'}})
print(f'拒否:     {w.error}')
print(f'          M1.name は無事: {M1.name!r}')

w.apply_edit({'op': 'move', 'target': 'M1', 'center': list(c0)})
print(f'復帰:     M1.center = {np.asarray(M1.center).round(4)}   (error={w.error!r})')
print()
print(f'受け取った編集: {len(w.edits)} 件')
print(f'編集可能な属性: {sorted(EDITABLE_OPTIC_ATTRS)}')

編集前:   M1.center = [ 0.5177 -0.0177]


move後:   M1.center = [ 0.5177 -0.0077]   (16 beams)
拒否:     EditError: 'name' is not an editable attribute of an optics.
          M1.name は無事: 'M1'
復帰:     M1.center = [ 0.5177 -0.0177]   (error='')

受け取った編集: 2 件
編集可能な属性: ['ARcenter', 'HRcenter', 'HRtransmissive', 'Refl_AR', 'Refl_HR', 'Trans_AR', 'Trans_HR', 'center', 'curve_direction', 'diameter', 'inv_ROC_AR', 'inv_ROC_HR', 'max_stray_order', 'n', 'normAngleHR', 'normVectHR', 'term_on_HR', 'term_on_HR_order', 'thickness', 'wedgeAngle']


### 素子の追加と削除

サイドバー上部の **`+ Mirror`** / **`+ CyMirror`** ボタンで、**現在の表示中心に**新しい素子を追加できます。追加された素子はすぐに選択状態になるので、そのままプロパティ欄で調整するかドラッグして配置できます。選択中の素子はプロパティパネル下部の **`Remove`** ボタンで削除できます。

新しい素子の既定値は、**レイアウトに既にある素子から引き継ぎます**（直径・厚み・ウェッジ角・屈折率・反射率）。10 cm のミラーが並ぶ系に足したら 25 cm（クラス既定値）ではなく 10 cm のミラーになる、という意図です。ただし**曲率は平面から始めます**（`inv_ROC_HR = 0`）。隣の素子の曲率が勝手に付いてくるのは便利ではなく驚きなので。`CyMirror` も同じ方針で `curve_direction='h'` の平面から始まります（種類は `Type` 欄と、`CyMirror` のときだけ現れる `Curve direction` 欄で分かります）。

名前は `M4`, `M5`, … / `CY1`, `CY2`, … のように未使用のものが自動で選ばれます。**名前を決めるのはJS側**です。そうすることで、シーンが返ってきた時点で新しい素子を名前で選択でき、Pythonからの返信チャネルが要りません。

```json
{"op": "add",    "type": "Mirror",   "name": "M4",
                 "params": {"HRcenter": [0.31, 0.22], "normAngleHR": 3.14159}}
{"op": "add",    "type": "CyMirror", "name": "CY1",
                 "params": {"HRcenter": [0.31, 0.22], "curve_direction": "h"}}
{"op": "remove", "target": "M4"}
```

`params` に渡せるキーもホワイトリスト（`CREATABLE_OPTIC_PARAMS`）で、`name` はここからは指定できません。`type` は `Mirror` と `CyMirror` のみ、`curve_direction` は `'h'` / `'v'` のみです（`ATTR_CHOICES`）。

In [15]:
# '+ Mirror' ボタンが送るのと同じメッセージ
w.apply_edit({'op': 'add', 'type': 'Mirror', 'name': 'M4',
              'params': {'HRcenter': [0.7, 0.2], 'normAngleHR': np.pi}})
M4 = layout.get_optics('M4')
print(f"追加後: 素子 {len(layout.optics)} 個, {len(w.scene['beams'])} beams")
print(f'  M4.diameter  = {float(M4.diameter):.3f} m  '
      f'(M3から継承。クラス既定値は 0.250 m)')
print(f'  M4.inv_ROC_HR = {float(M4.inv_ROC_HR):.3f}  (平面から始まる)')

# '+ CyMirror' ボタンが送るメッセージ
w.apply_edit({'op': 'add', 'type': 'CyMirror', 'name': 'CY1',
              'params': {'HRcenter': [0.3, -0.2], 'curve_direction': 'h'}})
CY1 = layout.get_optics('CY1')
print(f'  CY1: {type(CY1).__name__}, curve_direction={CY1.curve_direction!r}')

# 名前は自動でも決まる
w.apply_edit({'op': 'add', 'params': {'HRcenter': [0.2, -0.3]}})
print(f"自動命名: {[o.name for o in layout.optics]}")

# 'Remove' ボタンが送るメッセージ
for name in ['M4', 'M5', 'CY1']:
    w.apply_edit({'op': 'remove', 'target': name})
print(f"削除後: {[o.name for o in layout.optics]}, "
      f"{len(w.scene['beams'])} beams  (error={w.error!r})")

追加後: 素子 4 個, 16 beams
  M4.diameter  = 0.100 m  (M3から継承。クラス既定値は 0.250 m)
  M4.inv_ROC_HR = 0.000  (平面から始まる)
  CY1: CyMirror, curve_direction='h'
自動命名: ['M1', 'M2', 'M3', 'M4', 'CY1', 'M5']


削除後: ['M1', 'M2', 'M3'], 16 beams  (error='')


### ビーム幅の表示方法

サイドバーの **Beam width** パネルで、包絡線の描き方を選べます。

| 項目 | 選択肢 |
|---|---|
| Envelope | 1σ / 2.7σ (1 ppm) / 3σ |
| Direction | x / y / average |

**これまでの表示は x 方向でした。** `beam.draw()` の `mode` 既定値が `'x'` で、`layout.draw()` は `mode` を渡していなかったためです（平均ではありません）。今回 x / y / avg を選べるようにし、既定は従来どおり x のままにしました。ビームは一般に非点収差を持つので、どちらを見せるかは図が勝手に決められる話ではありません。

これらは**モデルではなく表示の設定**なので、`OpticalLayout.draw_options` に入ります。変更しても**再追跡は起きず、描き直しだけ**が走ります（`apply_edit` の `draw` 操作は `beams` を無効化しません）。

```json
{"op": "draw", "params": {"sigma_main": 1.0, "sigma_stray": 1.0, "width_mode": "y"}}
```

描画オプション一式は `gtrace.layout.DRAW_OPTIONS` に既定値付きで定義されています。`layout.draw(width_mode='y')` のように直接渡すこともでき、その場合は `draw_options` より優先されます。綴りを間違えたキーは `TypeError` で弾かれます（黙って無視されると、設定が効かない理由が分からなくなるため）。

なお静的HTML（8.）にはこのパネルがありません。描き直すには Python が要るからです。書き出す時点で選ぶことになります:

```python
layout.render_html('trace.html', width_mode='y', sigma_main=3, sigma_stray=3)
```

In [16]:
from gtrace.layout import DRAW_OPTIONS

# 非点収差のあるビームでないと x / y / avg の差が出ないので、
# ここだけ光源の y 方向のウェストを変えて確かめる。
b_ast = beam.GaussianBeam(q0=gauss.Rw2q(np.inf, 1*mm), wl=1064*nm,
                          pos=[0, 0], dirAngle=0, name='b0')
b_ast.qy = gauss.Rw2q(np.inf, 2.5*mm)
lay_ast = OpticalLayout(optics=[M1], sources=[b_ast],
                        rules=TraceRules(order=1, power_threshold=1e-3))
wx, wy = lay_ast.trace()[0].width(0.0)
print(f'ソースのビーム半径:  w_x = {wx*1e3:.3f} mm,  w_y = {wy*1e3:.3f} mm')
print()
print(f"{'mode':>5} {'1σ':>10} {'2.7σ':>10} {'3σ':>10}   [mm]")
for mode in ['x', 'y', 'avg']:
    row = []
    for s in [1.0, 2.7, 3.0]:
        cnv = lay_ast.draw(width_mode=mode, sigma_main=s, sigma_stray=s)
        row.append(abs(np.asarray(cnv.layers['main_beam_width']
                                  .shapes[0].y)[0]) * 1e3)
    print(f'{mode:>5} ' + ' '.join(f'{v:>10.3f}' for v in row))

print()
print(f"既定値: width_mode={DRAW_OPTIONS['width_mode']!r}, "
      f"sigma_main={DRAW_OPTIONS['sigma_main']}")

# 'Beam width' パネルが送るのと同じメッセージ（再追跡は起きない）
beams_before = w.scene['beams']
w.apply_edit({'op': 'draw', 'params': {'width_mode': 'avg'}})
print(f"draw操作後: display={ {k: w.scene['display'][k] for k in ['width_mode', 'sigma_main']} }, "
      f"ビーム数は {len(w.scene['beams'])} のまま")
w.apply_edit({'op': 'draw', 'params': {'width_mode': 'x'}})

ソースのビーム半径:  w_x = 1.000 mm,  w_y = 2.500 mm

 mode         1σ       2.7σ         3σ   [mm]
    x      1.000      2.700      3.000
    y      2.500      6.750      7.500
  avg      1.750      4.725      5.250

既定値: width_mode='x', sigma_main=2.7
draw操作後: display={'width_mode': 'avg', 'sigma_main': 2.7}, ビーム数は 16 のまま


True